# Занятие 1. Реальный мини-агент языковой лаборатории

**Цель практики:** собрать не теоретический пример, а маленького агента, который делает полезную работу для проекта сохранения малоресурсного языка.

Сценарий: у нас есть задача первичной разведки по удмуртскому языку. Агент должен:

1. найти реальные открытые материалы в Wikimedia Commons;
2. выбрать изображение, которое можно скачать;
3. прогнать OCR baseline в бесплатном Colab;
4. взять небольшой контрольный корпус из Удмуртской Википедии;
5. собрать отчет: что найдено, насколько читаемый OCR, что нужно проверить человеку.

Сначала соберем агента без фреймворка: обычные функции, словарь `state`, явные вызовы инструментов и решения. Потом завернем те же шаги в `LangGraph`, чтобы увидеть, зачем вообще нужен фреймворк для агентских workflow.

LLM/API-ключи не нужны: на первом занятии важнее понять архитектуру агента, чем подключать внешнюю модель.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-rus
!pip -q install langgraph pytesseract pillow pandas matplotlib requests

In [ ]:
import os, re, json, textwrap, math, statistics, random, io
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from typing import Any, Dict, List, TypedDict
from urllib.parse import quote
from PIL import Image, UnidentifiedImageError
from IPython.display import display
import matplotlib.pyplot as plt
import pytesseract
from langgraph.graph import StateGraph, END

## 1. Настройка реальных источников

Берем два открытых источника:

- Wikimedia Commons: категория `Udmurt Dunne`, где лежат реальные файлы, связанные с удмуртской газетой.
- Удмуртская Википедия: небольшой корпус страниц для сравнения с OCR-выводом.

Это не учебная таблица: агент будет обращаться к API и сохранять полученные данные.

In [ ]:
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
UDM_WIKI_API = 'https://udm.wikipedia.org/w/api.php'

COMMONS_CATEGORY = 'Category:Udmurt Dunne'
LANGUAGE = 'удмуртский'
FALLBACK_FILE_TITLE = 'File:Удномер.jpg'
RASTER_MIME_TYPES = {'image/jpeg', 'image/png', 'image/tiff', 'image/webp'}

def api_get(url, params, timeout=30):
    r = requests.get(url, params=params, timeout=timeout, headers={'User-Agent': 'lowres-course-colab/1.0'})
    r.raise_for_status()
    return r.json()

def commons_category_files(category, limit=20):
    data = api_get(COMMONS_API, {
        'action': 'query',
        'list': 'categorymembers',
        'cmtitle': category,
        'cmtype': 'file',
        'cmlimit': limit,
        'format': 'json',
    })
    return data.get('query', {}).get('categorymembers', [])

def commons_imageinfo(title):
    data = api_get(COMMONS_API, {
        'action': 'query',
        'titles': title,
        'prop': 'imageinfo',
        'iiprop': 'url|mime|size|extmetadata',
        'format': 'json',
    })
    pages = data.get('query', {}).get('pages', {})
    page = next(iter(pages.values()))
    info = page.get('imageinfo', [{}])[0]
    meta = info.get('extmetadata', {})
    return {
        'title': title,
        'url': info.get('url'),
        'mime': info.get('mime'),
        'width': info.get('width'),
        'height': info.get('height'),
        'license': meta.get('LicenseShortName', {}).get('value'),
        'artist': meta.get('Artist', {}).get('value'),
        'description': meta.get('ImageDescription', {}).get('value'),
    }

def is_raster_image(info):
    return bool(info.get('url')) and info.get('mime') in RASTER_MIME_TYPES

def fetch_udm_wiki_pages(limit=5):
    random_pages = api_get(UDM_WIKI_API, {
        'action': 'query',
        'generator': 'random',
        'grnnamespace': 0,
        'grnlimit': limit,
        'prop': 'extracts',
        'explaintext': 1,
        'exintro': 1,
        'format': 'json',
    })
    pages = []
    for page in random_pages.get('query', {}).get('pages', {}).values():
        title = page.get('title', '')
        extract = page.get('extract', '') or ''
        pages.append({
            'title': title,
            'chars': len(extract),
            'tokens': len(re.findall(r'\w+', extract.lower())),
            'cyrillic_share': cyrillic_share(extract),
            'url': 'https://udm.wikipedia.org/wiki/' + quote(title.replace(' ', '_')),
            'extract_preview': extract[:400],
        })
    return pages

def cyrillic_share(text):
    letters = re.findall(r'[A-Za-zА-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', text)
    if not letters:
        return 0.0
    cyr = [x for x in letters if re.match(r'[А-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', x)]
    return round(len(cyr) / len(letters), 3)

def text_diagnostics(text):
    tokens = re.findall(r'\w+', text.lower())
    short_tokens = [t for t in tokens if len(t) <= 2]
    weird = re.findall(r'[^\w\s.,:;!?()\-«»"\'/А-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', text)
    return {
        'chars': len(text),
        'tokens': len(tokens),
        'cyrillic_share': cyrillic_share(text),
        'short_token_share': round(len(short_tokens) / max(1, len(tokens)), 3),
        'weird_char_count': len(weird),
        'sample': text[:700],
    }

## 2. Агент без фреймворка: что под капотом

Минимальный агент состоит из четырех вещей:

- `state`: явная модель текущего контекста задачи;
- tools: функции, которые ходят во внешний мир или обрабатывают данные;
- observations: результаты вызова tools;
- policy: простые правила, которые решают, что делать дальше.

Ниже это обычный Python. Никакого LangGraph пока нет.

In [ ]:
def scout_sources(state):
    files = commons_category_files(state['commons_category'])
    if not files:
        files = [{'title': FALLBACK_FILE_TITLE, 'pageid': None}]
    state['commons_files'] = files
    save_artifact('lesson01_commons_candidates.csv', pd.DataFrame(files))
    return state

def choose_downloadable_image(state):
    inspected = []
    selected = None
    for item in state['commons_files']:
        info = commons_imageinfo(item['title'])
        inspected.append(info)
        if is_raster_image(info) and selected is None:
            selected = info
    if selected is None:
        selected = commons_imageinfo(FALLBACK_FILE_TITLE)
    state['selected_file'] = selected
    save_artifact('lesson01_commons_sources.csv', pd.DataFrame(inspected))
    return state

def download_openable_image(info, candidate_index=0):
    response = requests.get(
        info['url'],
        timeout=60,
        headers={'User-Agent': 'lowres-course-colab/1.0 (teaching notebook)'},
    )
    response.raise_for_status()
    content_type = response.headers.get('Content-Type', '').split(';')[0].strip().lower()
    if content_type and content_type not in RASTER_MIME_TYPES and content_type != 'application/octet-stream':
        raise ValueError(f'expected raster image, got Content-Type={content_type}')

    raw = response.content
    try:
        img = Image.open(io.BytesIO(raw)).convert('RGB')
    except UnidentifiedImageError as exc:
        raise ValueError('downloaded bytes are not an openable raster image') from exc

    if max(img.size) > 2200:
        img.thumbnail((2200, 2200))

    image_path = DATA_DIR / f'lesson01_ocr_source_{candidate_index}.jpg'
    img.save(image_path, format='JPEG', quality=92)
    return image_path

def run_ocr_baseline(state):
    selected_title = state['selected_file'].get('title')
    candidates = [state['selected_file']]
    for item in state['commons_files']:
        if item.get('title') != selected_title:
            candidates.append(commons_imageinfo(item['title']))

    errors = []
    image_path = None
    selected = None
    for i, candidate in enumerate(candidates):
        if not is_raster_image(candidate):
            errors.append({'title': candidate.get('title'), 'error': f"unsupported mime: {candidate.get('mime')}"})
            continue
        try:
            image_path = download_openable_image(candidate, i)
            selected = candidate
            break
        except Exception as exc:
            errors.append({'title': candidate.get('title'), 'error': str(exc)})

    if selected is None or image_path is None:
        save_artifact('lesson01_download_errors.csv', pd.DataFrame(errors))
        raise RuntimeError('Не удалось скачать ни одного открываемого raster-изображения из Commons candidates.')

    if errors:
        save_artifact('lesson01_download_errors.csv', pd.DataFrame(errors))

    text = pytesseract.image_to_string(Image.open(image_path), lang='rus')
    save_artifact('lesson01_ocr_raw.txt', text)

    state['selected_file'] = selected
    state['image_path'] = str(image_path)
    state['ocr_text'] = text
    return state

def decide_human_tasks(state):
    diag = text_diagnostics(state.get('ocr_text', ''))
    tasks = []
    if diag['chars'] < 100:
        tasks.append('OCR почти ничего не извлек: выбрать другой скан или улучшить предобработку.')
    if diag['cyrillic_share'] < 0.7:
        tasks.append('В тексте много некириллического шума: проверить язык OCR и качество изображения.')
    if diag['short_token_share'] > 0.45:
        tasks.append('Много коротких фрагментов: нужна ручная проверка строк и сегментации.')
    if not tasks:
        tasks.append('Выбрать 20-30 строк и вручную оценить ошибки OCR.')
    state['ocr_diagnostics'] = diag
    state['human_tasks'] = tasks
    return state

def add_wiki_probe(state):
    pages = fetch_udm_wiki_pages(limit=5)
    save_artifact('lesson01_udm_wiki_probe.csv', pd.DataFrame(pages))
    all_text = ' '.join(p['extract_preview'] for p in pages)
    state['wiki_pages'] = pages
    state['wiki_stats'] = text_diagnostics(all_text)
    return state

def build_agent_report(state):
    report = {
        'language': state['language'],
        'agent_type': state.get('agent_type', 'plain Python tool-using agent'),
        'real_sources': {
            'commons_category': state['commons_category'],
            'selected_file_title': state['selected_file'].get('title'),
            'selected_file_url': state['selected_file'].get('url'),
            'license': state['selected_file'].get('license'),
            'wiki_pages': [p['url'] for p in state['wiki_pages']],
        },
        'ocr_diagnostics': state['ocr_diagnostics'],
        'wiki_probe_stats': state['wiki_stats'],
        'next_human_tasks': state['human_tasks'],
        'state_fields_used': sorted(state.keys()),
    }
    state['report'] = report
    save_artifact('lesson01_real_agent_report.json', json.dumps(report, ensure_ascii=False, indent=2))
    return state

def run_plain_agent(language, commons_category):
    state = {
        'language': language,
        'commons_category': commons_category,
        'goal': 'оценить, можно ли начать OCR-разведку по открытым удмуртским материалам',
        'plan': [
            'найти файлы',
            'выбрать изображение',
            'запустить OCR',
            'оценить текст',
            'сравнить с wiki-корпусом',
            'собрать отчет',
        ],
    }
    for step in [
        scout_sources,
        choose_downloadable_image,
        run_ocr_baseline,
        decide_human_tasks,
        add_wiki_probe,
        build_agent_report,
    ]:
        state = step(state)
        print('done:', step.__name__, '| state keys:', sorted(state.keys()))
    return state

plain_state = run_plain_agent(LANGUAGE, COMMONS_CATEGORY)
print(json.dumps(plain_state['report'], ensure_ascii=False, indent=2))

## 3. Та же логика в LangGraph

Теперь берем те же функции и раскладываем их в граф. Смысл LangGraph не в том, что он “умнее”, а в том, что он делает архитектуру явной:

- у каждого узла есть входной и выходной `state`;
- переходы между шагами видны отдельно от кода инструментов;
- позже можно добавлять условия, повторы, LLM-узлы, human review и сохранение состояния между запусками.

In [ ]:
class LabAgentState(TypedDict, total=False):
    language: str
    commons_category: str
    commons_files: List[Dict[str, Any]]
    selected_file: Dict[str, Any]
    image_path: str
    ocr_text: str
    ocr_diagnostics: Dict[str, Any]
    wiki_pages: List[Dict[str, Any]]
    wiki_stats: Dict[str, Any]
    report: Dict[str, Any]
    human_tasks: List[str]
    agent_type: str

def source_scout_node(state: LabAgentState):
    next_state = scout_sources(dict(state))
    return {'commons_files': next_state['commons_files']}

def metadata_node(state: LabAgentState):
    next_state = choose_downloadable_image(dict(state))
    return {'selected_file': next_state['selected_file']}

def ocr_node(state: LabAgentState):
    next_state = run_ocr_baseline(dict(state))
    return {'image_path': next_state['image_path'], 'ocr_text': next_state['ocr_text']}

def ocr_diagnostics_node(state: LabAgentState):
    next_state = decide_human_tasks(dict(state))
    return {'ocr_diagnostics': next_state['ocr_diagnostics'], 'human_tasks': next_state['human_tasks']}

def wiki_probe_node(state: LabAgentState):
    next_state = add_wiki_probe(dict(state))
    return {'wiki_pages': next_state['wiki_pages'], 'wiki_stats': next_state['wiki_stats']}

def report_node(state: LabAgentState):
    next_state = build_agent_report(dict(state))
    return {'report': next_state['report']}

workflow = StateGraph(LabAgentState)
workflow.add_node('source_scout', source_scout_node)
workflow.add_node('metadata', metadata_node)
workflow.add_node('ocr', ocr_node)
workflow.add_node('ocr_diagnostics', ocr_diagnostics_node)
workflow.add_node('wiki_probe', wiki_probe_node)
workflow.add_node('report', report_node)

workflow.set_entry_point('source_scout')
workflow.add_edge('source_scout', 'metadata')
workflow.add_edge('metadata', 'ocr')
workflow.add_edge('ocr', 'ocr_diagnostics')
workflow.add_edge('ocr_diagnostics', 'wiki_probe')
workflow.add_edge('wiki_probe', 'report')
workflow.add_edge('report', END)

agent = workflow.compile()

## 4. Запускаем LangGraph-версию

In [ ]:
state = agent.invoke({
    'language': LANGUAGE,
    'commons_category': COMMONS_CATEGORY,
    'agent_type': 'LangGraph tool-using workflow agent',
})

print(json.dumps(state['report'], ensure_ascii=False, indent=2))

## 5. Смотрим реальные артефакты

In [ ]:
print('Выбранный файл:', state['selected_file']['title'])
print('Лицензия:', state['selected_file'].get('license'))
print('URL:', state['selected_file'].get('url'))

img = Image.open(state['image_path'])
print('Размер изображения для OCR:', img.size)
display(img)

In [ ]:
print(state['ocr_text'][:2000])

In [ ]:
display(pd.DataFrame(state['wiki_pages'])[['title', 'chars', 'tokens', 'cyrillic_share', 'url']])
display(pd.DataFrame([state['ocr_diagnostics'], state['wiki_stats']], index=['ocr', 'wiki_probe']))

## 6. Что здесь агентского

В этой вводной тетрадке важны не “виды агентов”, а архитектура на конкретной задаче:

- `state` хранит текущую картину задачи: цель, найденные источники, выбранный файл, OCR-текст, диагностику, контрольный корпус и задачи для человека.
- каждый узел делает один проверяемый шаг;
- следующий шаг выбирается на основании уже собранного состояния;
- человек остается в контуре там, где нельзя автоматически решать про права, качество и публикацию.

В следующих тетрадках мы уже не будем каждый раз подробно разбирать архитектуру. Будем использовать этот принцип как рабочий шаблон: источник, tool, state, диагностика, human boundary, отчет.

## Вопросы для отчета

1. Какой реальный материал нашел агент и можно ли понять его лицензию?
2. Какие части plain Python-агента соответствуют `state`, tools, observations и policy?
3. Что стало понятнее или надежнее после переноса той же логики в LangGraph?
4. Получился ли OCR-текст вменяемым? Покажите 3-5 характерных ошибок.
5. Что в этой задаче нельзя отдавать агенту полностью автоматически?